# Model-Form Uncertainty — Inferring Physics Trust

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/07_model_form_uncertainty.ipynb)

This notebook reproduces **Example 2, Figure 2** from Alberts & Bilionis (2023).

Consider the nonlinear PDE

$$D\phi''(x) - \kappa\phi^3(x) = f(x), \quad x \in [0,1]$$

with zero Dirichlet BCs.  We allow the physics model to be **misspecified** via a
correctness parameter $\gamma \in [0, 1]$:

- $\gamma = 1$: correct physics (source or energy functional matches ground truth).
- $\gamma = 0$: completely wrong physics.

**Algorithm 3** (nested SGLD) jointly infers $\lambda = \log\beta$ (the log physics-trust
parameter) alongside the field $\phi$ via two interleaved SGLD chains.  As model-form
correctness $\gamma$ degrades, the posterior over $\beta$ shifts toward smaller values,
reflecting reduced confidence in the physics.

Two experiment types are run:
- **source_error:** misspecified source $f_\gamma(x) = \gamma\cos(4x) + (1-\gamma)e^{-x}$.
- **energy_error:** misspecified energy functional (mixing $\phi^4$ with $\phi^2$ terms).

**Estimated runtime:** 15–30 min on CPU with the default single-$\gamma$ CONFIG.
Expand `gamma_values` to `[0.0, 0.5, 1.0]` for the full sweep (adds 2× runtime per
experiment type).

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git

import time
import jax
jax.config.update('jax_enable_x64', True)
import numpy as np
import matplotlib.pyplot as plt

from pipelines.phase_b_model_form import run_phase_b_model_form

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    'D':                  0.1,
    'kappa':              1.0,
    'n_obs':              40,
    'noise_std':          0.01,
    'gamma_values':       [1.0],    # expand to [0.0, 0.5, 1.0] for full sweep
    'K':                  20,
    'warmup_steps':       3000,     # inner-only warm-up                     [100, 1000000]
    'outer_steps':        1500,     # outer SGLD on lambda=log(beta)         [100, 20000]
    'outer_step_size0':   1e-3,     # outer alpha_0                          [1e-7, 1e-2]
    'inner_step_size0':   5e-6,     # inner alpha_0 approx 1/beta_0         [1e-7, 1e-2]
    'n_quad':             1,        # 1-point stochastic estimator           [1, 256]
    'n_grid':             200,
    'burn_in_frac':       0.3,
}

## Run Nested SGLD

In [ ]:
t_start = time.perf_counter()

mf_result = run_phase_b_model_form(
    cfg=CONFIG,
    device_preference=jax.default_backend(),
    save_outputs=False,
)

elapsed = time.perf_counter() - t_start
print(f'Status:  {mf_result["status"]}')
print(f'Runtime: {elapsed:.1f} s  ({elapsed/60:.1f} min)')

## Results

In [ ]:
experiments = mf_result.get('experiments', {})

print('Posterior beta summary by experiment type and gamma:')
for exp_type in ('source_error', 'energy_error'):
    runs = experiments.get(exp_type, [])
    if not runs:
        continue
    print(f'  [{exp_type}]')
    for entry in runs:
        gamma   = entry['gamma']
        med     = entry['beta_median']
        lo      = entry['beta_q05']
        hi      = entry['beta_q95']
        stopped = entry.get('stopped_early', False)
        flag    = '  (stopped early)' if stopped else ''
        print(f'    gamma={gamma:.2f}   beta median={med:11.4g}   [90% CI: {lo:11.4g}, {hi:11.4g}]{flag}')

# Optional: plot log(beta) trace for each run
all_runs = []
for exp_type in ('source_error', 'energy_error'):
    for entry in experiments.get(exp_type, []):
        log_chain = entry.get('log_beta_chain')
        if log_chain is not None and len(log_chain) > 0:
            all_runs.append((exp_type, entry['gamma'], np.asarray(log_chain)))

if all_runs:
    fig, axes = plt.subplots(1, len(all_runs),
                             figsize=(5 * len(all_runs), 3.5),
                             squeeze=False)
    axes = axes[0]
    for ax, (exp_type, gamma, chain) in zip(axes, all_runs):
        ax.plot(chain, lw=0.8, color='steelblue', alpha=0.9)
        ax.axhline(np.median(chain[int(0.3*len(chain)):]),
                   color='tomato', lw=1.5, linestyle='--', label='post-burn-in median')
        ax.set_xlabel('Outer SGLD step')
        ax.set_ylabel('log(\u03b2)')
        ax.set_title(f'{exp_type}\n\u03b3={gamma:.2f}', fontsize=10)
        ax.legend(fontsize=8)
    fig.suptitle('log(\u03b2) Chain Trace', fontsize=12)
    fig.tight_layout()
    show_fig(fig)
else:
    print('No log_beta_chain data available for plotting.')

## Interpretation

**Expected behavior** when `gamma_values = [0.0, 0.5, 1.0]`:

- $\gamma = 1$ (correct physics): the posterior over $\beta$ concentrates at large values
  ($\beta \gg 1$), reflecting high trust in the physics.
- $\gamma \to 0$ (wrong physics): the chain drifts toward smaller $\beta$, because fitting
  the observations with the wrong PDE requires less physics regularisation.

This notebook currently runs a **single** $\gamma = 1.0$ for speed.  Expand
`gamma_values` in the CONFIG above to reproduce the full Figure 2 sweep.

The log($\beta$) trace plot provides a stationarity check: the chain should stabilise
after the warm-up phase.  If it is still drifting at the end of the run, increase
`outer_steps` or reduce `outer_step_size0`.